In [1]:
import os
import json
import re
import sqlite3
import pandas as pd
import numpy as np

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

### Создание таблицы projects

In [7]:
os.makedirs("../db/mos_estate", exist_ok=True)
conn = sqlite3.connect(r'../db/mos_estate/mos_estate.sqlite3')
cursor = conn.cursor()

In [8]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS projects (
    id INTEGER PRIMARY KEY,
    developer TEXT NOT NULL,
    project TEXT NOT NULL,
    class TEXT,
    district TEXT,
    metro TEXT,
    dist_to_metro REAL,
    num_residents INTEGER,
    shcool_plan INTEGER,
    shcool_fact INTEGER,
    nursery_plan INTEGER,
    nursery_fact INTEGER,
    parking TEXT
)
''')

### Заполнение таблицы projects

In [9]:
data_to_insert = []

In [10]:
with open(r'../raw_data/general_data.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        record = json.loads(line.strip())
        row = (
            record.get('developer'),
            record.get('project'),
            record.get('class'),
            record.get('district'),
            record.get('metro'),
            record.get('dist_to_metro'),
            record.get('num_residents'),
            record.get('shcool_plan'),
            record.get('shcool_fact'),
            record.get('nursery_plan'),
            record.get('nursery_fact'),
            record.get('parking')
        )
        data_to_insert.append(row)

In [11]:
try:
    cursor.execute('BEGIN TRANSACTION')
    
    cursor.executemany('''
        INSERT INTO projects (
            developer, project, class, district, metro, dist_to_metro,
            num_residents, shcool_plan, shcool_fact, nursery_plan, nursery_fact, parking
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', data_to_insert)
    
    conn.commit()
    print(f"Успешно вставлено {len(data_to_insert)} записей")
    
except Exception as e:
    conn.rollback()
    print(f"Ошибка при вставке: {e}")

finally:

    cursor.execute('PRAGMA synchronous = NORMAL')
    cursor.execute('PRAGMA journal_mode = DELETE')

Успешно вставлено 96 записей


### Создание таблицы descriptions

In [12]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS descriptions (
    id INTEGER PRIMARY KEY,
    id_project INTEGER NOT NULL,
    description TEXT
)
''')

### Заполнение таблицы description

In [13]:
data_to_insert = []

In [14]:
with open(r'../raw_data/descriptions.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        record = json.loads(line.strip())
        row = (
            record.get('id_project'),
            record.get('description')
        )
        data_to_insert.append(row)

In [15]:
try:
    cursor.execute('BEGIN TRANSACTION')
    
    cursor.executemany('''
        INSERT INTO descriptions (
            id_project, description
        ) VALUES (?, ?)
    ''', data_to_insert)
    
    conn.commit()
    print(f"Успешно вставлено {len(data_to_insert)} записей")
    
except Exception as e:
    conn.rollback()
    print(f"Ошибка при вставке: {e}")

finally:
    cursor.execute('PRAGMA synchronous = NORMAL')
    cursor.execute('PRAGMA journal_mode = DELETE')

Успешно вставлено 96 записей


In [16]:
conn.close()

### Очистка данных

In [17]:
conn = sqlite3.connect(r'../db/mos_estate/mos_estate.sqlite3')
cursor = conn.cursor()

In [18]:
df = pd.read_sql_query("SELECT * FROM descriptions", conn)

In [19]:
def text_cleaner(text):

    # Конвертация в str
    text = str(text)
    
    # Преобразование текста в нижний регистр
    text = text.lower()

    # Удаление лишних пробелов
    #text = re.sub(r'\s+', ' ', text)

    # Удаление знаков препинания
    text = re.sub(r'[^\w\s]', '', text)

    # Удаление стоп-слов
    stop_words = set(stopwords.words('russian'))
    tokens = word_tokenize(text)
    filtered_text = [word for word in tokens if word.lower() not in stop_words]
    
    # Объединение лемматизированных слов обратно в текст
    cleaned_text = ' '.join(filtered_text)

    return cleaned_text

In [20]:
df['cleaned_description'] = df['description'].map(lambda description: text_cleaner(description))

In [21]:
df.head()

,id,id_project,description,cleaned_description
0,1,1,«Ильинские луга» — масштабный проект с собстве...,ильинские луга масштабный проект собственной и...
1,2,2,«Белая Дача парк» — проект в Московской област...,белая дача парк проект московской области горо...
2,3,3,Жилой район «Саларьево парк» расположен в Ново...,жилой район саларьево парк расположен новой мо...
3,4,4,Жилой квартал «Люблинский парк» расположен на ...,жилой квартал люблинский парк расположен югово...
4,5,5,«Зелёный парк» — масштабный проект в экологичн...,зелёный парк масштабный проект экологичном рай...


### Запись очищенных данных в бд

In [22]:
cursor.execute('ALTER TABLE descriptions ADD COLUMN cleaned_description TEXT')
conn.commit()

In [23]:
rows = list(zip(df['cleaned_description'], df['id_project']))

In [24]:
update_query = '''
UPDATE descriptions
SET cleaned_description = ?
WHERE id_project = ?
'''

In [25]:
cursor.executemany(update_query, rows)
conn.commit()
print(f"Обновлено строк: {cursor.rowcount}")

Обновлено строк: 96


In [26]:
conn.commit()

In [27]:
conn.close()

### Подготовка чанков

In [2]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [3]:
conn = sqlite3.connect(r'../db/mos_estate/mos_estate.sqlite3')
cursor = conn.cursor()

cursor.execute(
    """  SELECT
    p.developer,
    p.project,
    p.class,
    p.district,
    p.metro,
    p.dist_to_metro,
    p.num_residents,
    p.shcool_plan,
    p.shcool_fact,
    p.nursery_plan,
    p.nursery_fact,
    p.parking,
    d.cleaned_description
        FROM projects AS p
        JOIN descriptions AS d
        ON p.id = d.id_project;
"""
)
all_rows = cursor.fetchall()
conn.close()

In [4]:
all_docs = []
for row in all_rows:
    context_lines = [
            f"Застройщик: {row[0]}",
            f"Название ЖК: {row[1]}",
            f"Класс жилья: {row[2]}",
            f"Район: {row[3]}",
            f"Метро: {row[4]}",
            f"Расстояние до метро в км: {row[5]}",
            f"Количество жильцов: {row[6]}",
            f"Планируемое количество мест в школе: {row[7]}",
            f"Доступных мест в школе сейчас: {row[8]}",
            f"Планируемое количество мест в детсадах: {row[9]}",
            f"Доступных мест в детсадах сейчас: {row[10]}",
            f"Тип парковки: {row[11]}",
            f"Контекст из описания ЖК {row[12]}",
        ]
    context = '\n'.join(context_lines)
    all_docs.append(Document(page_content=context))

In [5]:
len(all_docs)

96

In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunked_docs = splitter.split_documents(all_docs)

In [7]:
print(f"Было документов: {len(all_docs)}, стало чанков: {len(chunked_docs )}")

Было документов: 96, стало чанков: 606


### Создание векторной базы

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate

In [12]:
persist_directory = "../db/chroma"
os.makedirs(persist_directory, exist_ok=True)

In [13]:
#model_name = "Qwen/Qwen3-Embedding-0.6B"
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={
        "device": "cpu",
    },
    encode_kwargs={
        "normalize_embeddings": True,
    },
)
vectorstore = Chroma.from_documents(
    documents=chunked_docs,
    embedding=embeddings,
    persist_directory=persist_directory
)

In [14]:
#  Удаление БД
#import shutil
#shutil.rmtree('../db', ignore_errors=True)

### Реализация Context Expansion

In [22]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import LocalFileStore, create_kv_docstore

In [30]:
store = create_kv_docstore(LocalFileStore("../db/parent_docs_store"))

In [24]:
vectorstore = Chroma(
    collection_name="rag_children",
    embedding_function=embeddings, 
    persist_directory="../db/chroma_children",
)

In [25]:
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50),
    parent_splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100),
    search_kwargs={"k": 5},
)

In [26]:
retriever.add_documents(all_docs)

### Проверка векторной базы

In [14]:
import chromadb

client = chromadb.PersistentClient(path='../db/chroma')
print(client.list_collections())

[Collection(name=langchain)]


In [15]:
import chromadb
client = chromadb.PersistentClient(path='../db/chroma')
col = client.get_collection('langchain')
print(col.count())  # должно быть > 0

462


In [15]:
import chromadb
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

persist_dir = '../db/chroma'
collection_name_candidate = 'langchain'  # твоё имя

# 1. Смотрим, какие коллекции реально есть
client = chromadb.PersistentClient(path=persist_dir)
collections = client.list_collections()
print("Доступные коллекции:", [c.name for c in collections])

# 2. Проверяем count хотя бы одной коллекции
for c in collections:
    print(f"Коллекция '{c.name}': count = {c.count()}")

# 3. Выбираем правильное имя (если default_collection нет — берём langchain или другое)
if collection_name_candidate not in [c.name for c in collections]:
    print("Коллекция", collection_name_candidate, "не найдена. Используем первую доступную.")
    collection_name = collections[0].name if collections else None
else:
    collection_name = collection_name_candidate

if not collection_name:
    raise ValueError("Не найдено ни одной коллекции в указанной директории.")

# 4. Подключаемся с тем же embedding, что и при создании
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

collection_name = "langchain"
vectorstore2 = Chroma(
    persist_directory=persist_dir,
    embedding_function=embeddings,
    collection_name=collection_name,
)

# 5. Делаем поиск
query = "зоны для прогулок и игр"  # исправленная переменная
docs2 = vectorstore2.similarity_search(query, k=5)
print("Найдено документов:", len(docs2))
if docs2:
    for d in docs2:
        print("---")
        print(d.metadata)
        print(d.page_content[:200])
else:
    print("Результат пустой — проверь эмбеддинги и имя коллекции.")

/tmp/ipykernel_9385/379036811.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
/tmp/ipykernel_9385/379036811.py:28: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Доступные коллекции: ['langchain']
Коллекция 'langchain': count = 606
Найдено документов: 5
---
{}
предусмотрели квартиры окнами пол угловыми окнами также французскими открытыми закрытыми балконами
---
{}
окна пол эркеры открытые балконы террасы квартиры сдаются предчистовой отделкой
---
{}
новоселья парковка жителей гостей предусмотрен плоскостной паркинг
---
{}
плоскостные паркинги обеспечивают место каждого автомобиля
---
{}
пассажирский грузовой лифты парковки машину оставить подземном многоуровневом наземном паркинге подземных парковках предусмотрены зарядки электромобилей


### Проверка ретривера Context Expansion

In [27]:
user_query = "Я ищу квартиру в проекте ПИК, где для моих детей обязательно должна быть масштабная игровая площадка типа «плейхаб гравити» или «плейхаб ферма». Какие проекты подходят под это требование?"

In [28]:
results = retriever.invoke(user_query)

In [29]:
results

[Document(metadata={'dev': 'ПИК', 'proj': 'Люблинский парк', 'h_class': 'Комфорт', 'distr': 'Люблино', 'metro': 'Братиславская', 'dist_to_metro': 2.5, 'num_resid': 30000, 's_plan': 4500, 's_fact': 1150, 'n_plan': 1800, 'n_fact': 675, 'parking': 'Подземный'}, page_content='третьим проектом пик большой игровой площадкой плейхаб которая открылась 1м 2м блоками безопасном расстоянии дороги новое детское пространство остров воплощает идею бесконечной игры ландшафт площадки это ещё игровой элемент который помогает детям придумывать множество самых разнообразных игр интересно детям разного возраста малышей подростков жителям квартала люблинский парк друзьям доступен проект пикспорт это бесплатные занятия профессиональными тренерами тренировки разным направлениям например йога фитнес бег проходят группе улице рядом домом онлайн'),
 Document(metadata={'dev': 'ПИК', 'proj': 'Люблинский парк', 'h_class': 'Комфорт', 'distr': 'Люблино', 'metro': 'Братиславская', 'dist_to_metro': 2.5, 'num_resid': 3